# Experiment validation

These tests use small fixtures to check audio handling, split isolation, model behavior and scoring. Run the notebook in the full training environment to execute the suite.

In [ ]:
from pathlib import Path
import os
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks/code").is_dir() and (p / "results/fits.json").is_file())
os.chdir(root)
get_ipython().run_line_magic("run", '"notebooks/00_runtime.ipynb"')
__name__ = "_mimii_validation_notebook"
sys.modules[__name__] = sys.modules["__main__"]

## Archive integrity

In [ ]:
__file__ = str(root / "tests/test_download_mimii.py")

In [ ]:
"""Small synthetic fixtures for archive integrity and resumable downloading."""

import hashlib
import importlib.util
import io
from pathlib import Path
import tempfile
import unittest
from unittest.mock import patch
from zipfile import ZipFile


spec = importlib.util.spec_from_file_location(
    "download_mimii", Path(__file__).resolve().parents[1] / "scripts/download_mimii.py"
)
downloader = importlib.util.module_from_spec(spec)
spec.loader.exec_module(downloader)

class DownloadTests(unittest.TestCase):
    def setUp(self):
        self.temp = tempfile.TemporaryDirectory()
        self.addCleanup(self.temp.cleanup)
        self.data = Path(self.temp.name)
        (self.data / "metadata").mkdir()

    def archive(self, filename="pump/id_00/normal/00000000.wav"):
        archive = self.data / "0_dB_pump.zip"
        with ZipFile(archive, "w") as package:
            package.writestr(filename, b"synthetic audio fixture")
        return {
            "key": archive.name, "size": archive.stat().st_size,
            "checksum": "md5:" + hashlib.md5(archive.read_bytes()).hexdigest(),
            "links": {"self": "https://example.invalid/archive"},
        }

    def test_extract_verify_and_preserve_existing_archive(self):
        entry = self.archive()
        downloader.process(entry, self.data)
        self.assertEqual((self.data / "pump_0db/pump/id_00/normal/00000000.wav").read_bytes(),
                         b"synthetic audio fixture")
        self.assertTrue((self.data / entry["key"]).exists())
        downloader.process(entry, self.data)

    def test_reject_bad_checksum_before_extraction(self):
        entry = self.archive()
        entry["checksum"] = "md5:" + "0" * 32
        with self.assertRaisesRegex(ValueError, "Checksum mismatch"):
            downloader.process(entry, self.data)
        self.assertFalse((self.data / "pump_0db").exists())

    def test_reject_archive_traversal(self):
        entry = self.archive("pump/../../outside.wav")
        with self.assertRaisesRegex(ValueError, "Unexpected archive path"):
            downloader.process(entry, self.data)
        self.assertFalse((self.data / "outside.wav").exists())

    def test_preserve_different_existing_recording(self):
        entry = self.archive()
        existing = self.data / "pump_0db/pump/id_00/normal/00000000.wav"
        existing.parent.mkdir(parents=True)
        existing.write_bytes(b"keep this recording")
        with self.assertRaisesRegex(ValueError, "Existing file differs"):
            downloader.process(entry, self.data)
        self.assertEqual(existing.read_bytes(), b"keep this recording")

    def test_resume_contiguous_prefix(self):
        archive = self.data / "archive.part"
        archive.write_bytes(b"abc")

        def response(request, timeout):
            self.assertEqual(request.headers["Range"], "bytes=3-5")
            result = io.BytesIO(b"def")
            result.status = 206
            result.headers = {"Content-Range": "bytes 3-5/6"}
            return result

        with patch.object(downloader, "urlopen", response):
            downloader.download({"key": "archive", "size": 6,
                                 "links": {"self": "https://example.invalid/archive"}}, archive)
        self.assertEqual(archive.read_bytes(), b"abcdef")

if __name__ == "__main__":
    unittest.main()

## Audio, labels and split isolation

In [ ]:
__file__ = str(root / "tests/test_preprocessing.py")

In [ ]:
"""Small synthetic fixtures; compatible with pytest and stdlib unittest."""

from dataclasses import replace
import contextlib
import io
import json
from pathlib import Path
import tempfile
import unittest
from unittest.mock import patch

import numpy as np
import soundfile as sf

from mimii.audio import load_audio
from mimii.config import PreprocessConfig
from mimii.dataset import assign_reference_splits, assign_splits, discover_recordings, parse_recording, validate_splits
from mimii.features import context_vectors, fit_normalizer, log_mel_spectrogram, normalize

class PreprocessingTests(unittest.TestCase):
    def setUp(self):
        self.temporary = tempfile.TemporaryDirectory()
        self.root = Path(self.temporary.name)
        self.config = PreprocessConfig(duration_seconds=0.1, context_frames=3, workers=1)

    def tearDown(self):
        self.temporary.cleanup()

    def write_audio(self, data, sample_rate=16000, name="fixture.wav"):
        path = self.root / name
        path.parent.mkdir(parents=True, exist_ok=True)
        sf.write(path, data, sample_rate, subtype="PCM_16")
        return path

    def test_first_microphone_preserves_amplitude(self):
        audio = np.column_stack((np.full(1600, 0.25), np.full(1600, -0.5)))
        path = self.write_audio(audio)
        first, meta = load_audio(path, self.config)
        second, _ = load_audio(path, replace(self.config, channel=1))
        np.testing.assert_allclose(first, 0.25)
        np.testing.assert_allclose(second, -0.5)
        self.assertEqual(meta["channels"], 2)
        self.assertAlmostEqual(meta["rms"], 0.25)

    def test_malformed_audio_is_rejected(self):
        for path, config in (
            (self.write_audio(np.ones(1600), sample_rate=8000, name="wrong_rate.wav"), self.config),
            (self.write_audio(np.ones(800), name="short.wav"), self.config),
            (self.write_audio(np.ones(1600), name="mono.wav"), replace(self.config, channel=1)),
        ):
            with self.assertRaises(ValueError):
                load_audio(path, config)

    def test_labels_and_id_parsing(self):
        for condition, label in (("normal", 0), ("abnormal", 1)):
            row = parse_recording(self.root / "id_02" / condition / "original.wav", self.root)
            self.assertEqual(row["machine_id"], "id_02")
            self.assertEqual(row["label"], label)
            self.assertEqual(row["filename"], "original.wav")
        for relative in ("id_2/normal/x.wav", "id_02/unknown/x.wav", "normal/x.wav"):
            with self.assertRaises(ValueError):
                parse_recording(self.root / relative, self.root)
        with self.assertRaises(ValueError):
            discover_recordings(self.root)

    def test_logmel_shape_and_silence_floor(self):
        config = PreprocessConfig()
        silent = log_mel_spectrogram(np.zeros(160000, dtype=np.float32), config)
        self.assertEqual(silent.shape, (64, 313))
        self.assertEqual(context_vectors(silent, 5).shape, (309, 320))
        self.assertTrue(np.isfinite(silent).all())
        np.testing.assert_allclose(silent, 10 * np.log10(np.finfo(np.float64).eps))

    def test_logmel_retains_absolute_energy(self):
        t = np.arange(1600) / 16000
        y = (0.2 * np.sin(2 * np.pi * 1000 * t)).astype(np.float32)
        a = log_mel_spectrogram(y, self.config)
        b = log_mel_spectrogram(y * 0.5, self.config)
        active = a > -60
        np.testing.assert_allclose((a - b)[active], 20 * np.log10(2), atol=0.01)

    def test_context_order_and_validation(self):
        feature = np.array([[1, 2, 3], [10, 20, 30]], dtype=np.float32)
        np.testing.assert_array_equal(context_vectors(feature, 2), [[1, 10, 2, 20], [2, 20, 3, 30]])
        for frames in (0, 4):
            with self.assertRaises(ValueError):
                context_vectors(feature, frames)

    def synthetic_records(self):
        rows = []
        for mid in ("id_00", "id_02"):
            for label, count in ((0, 20), (1, 3)):
                for index in range(count):
                    path = f'{mid}/{"abnormal" if label else "normal"}/{index}.wav'
                    rows.append({"relative_path": path, "machine_id": mid, "label": label,
                                 "waveform_sha256": f"{mid}:{label}:{index}"})
        return rows

    def test_split_isolation_and_determinism(self):
        original = self.synthetic_records()
        duplicate = dict(original[0], relative_path="id_00/normal/duplicate.wav")
        original.append(duplicate)
        first = assign_splits([dict(r) for r in original], seed=42)
        second = assign_splits([dict(r) for r in reversed(original)], seed=42)
        mapping = lambda rows: {r["relative_path"]: r["split"] for r in rows}
        self.assertEqual(mapping(first), mapping(second))
        self.assertEqual(first[0]["split"], first[-1]["split"])
        self.assertTrue(all(r["split"] == "test" for r in first if r["label"] == 1))
        third = assign_splits([dict(r) for r in original], seed=99)
        self.assertNotEqual(mapping(first), mapping(third))
        validate_splits(first)

    def test_duplicate_label_conflict_rejected(self):
        rows = self.synthetic_records()
        rows[-1]["waveform_sha256"] = rows[0]["waveform_sha256"]
        with self.assertRaises(ValueError):
            assign_splits(rows)

    def test_reference_split_survives_changed_waveforms_at_other_snr(self):
        reference = assign_splits(self.synthetic_records())
        for row in reference:
            row.update(machine_type="fan", condition="abnormal" if row["label"] else "normal")
        target = [dict(row, waveform_sha256="new_snr:" + row["waveform_sha256"]) for row in reference]
        for row in target:
            row.pop("split")
        assign_reference_splits(target, reference)
        self.assertEqual([r["split"] for r in target], [r["split"] for r in reference])
        with self.assertRaises(ValueError):
            assign_reference_splits(target[:-1], reference)
        target[0]["label"] = 1
        with self.assertRaises(ValueError):
            assign_reference_splits(target, reference)

    def test_normalizer_ignores_validation_and_test(self):
        rows = []
        for index, (split, value, label) in enumerate((("train", 1., 0), ("train", 3., 0),
                                                      ("validation", 1000., 0), ("test", -1000., 1))):
            path = f"{index}.npy"
            np.save(self.root / path, np.full((2, 3), value, dtype=np.float32))
            rows.append({"split": split, "label": label, "feature_path": path})
        stats = fit_normalizer(rows, self.root)
        np.testing.assert_array_equal(stats["mean"], [2, 2])
        np.testing.assert_array_equal(stats["std"], [1, 1])
        self.assertEqual(stats["n_frames"], 6)
        self.assertEqual(stats["n_clips"], 2)
        np.testing.assert_array_equal(normalize(np.ones((2, 3)), stats["mean"], stats["std"]), -np.ones((2, 3)))

    def test_config_rejects_invalid_parameters(self):
        for kwargs in ({"n_fft": 0}, {"channel": -1}, {"fmax": 9000},
                       {"validation_fraction": 0.8, "test_normal_fraction": 0.3}):
            with self.assertRaises(ValueError):
                PreprocessConfig(**kwargs)

    def test_end_to_end_manifest_and_output_guard(self):
        from mimii.preprocess import run

        data_root = self.root / "fan"
        rng = np.random.default_rng(10)
        for mid in ("id_00", "id_02"):
            for index in range(6):
                condition = "abnormal" if index == 5 else "normal"
                self.write_audio(rng.normal(0, 0.1, (1600, 2)),
                                 name=f"fan/{mid}/{condition}/{index:08}.wav")
        output = self.root / "output"
        config = replace(self.config, data_root=str(data_root), output_dir=str(output))
        with patch("mimii.visualization.create_figures"), contextlib.redirect_stdout(io.StringIO()):
            summary = run(config)
        self.assertEqual(summary["recordings"], 12)
        self.assertEqual(summary["split_counts"], {"train": 6, "validation": 2, "test": 4})
        self.assertEqual(json.loads((output / "status.json").read_text())["state"], "complete")
        rows = [json.loads(line) for line in (output / "manifest.jsonl").read_text().splitlines()]
        validate_splits(rows)
        self.assertEqual(len(list((output / "logmel").rglob("*.npy"))), 12)
        self.assertTrue((output / "normalizer.npz").exists())
        with self.assertRaises(FileExistsError):
            run(config)

if __name__ == "__main__":
    unittest.main()

## Matching recordings across noise levels

In [ ]:
__file__ = str(root / "tests/test_multisnr.py")

In [ ]:
"""Prevent leakage through differently noised copies of a source recording."""

from copy import deepcopy
from pathlib import Path
from types import SimpleNamespace
import unittest

from mimii.dataset import validate_splits
from mimii.prepare_multisnr import FRONTEND_KEYS, combine_records
from mimii.report_multisnr import statistics

class MultiSnrTests(unittest.TestCase):
    def test_seed_variability_is_population_variance_not_standard_error(self):
        result = statistics([0.6, 0.7, 0.8])
        self.assertAlmostEqual(result["mean"], 0.7)
        self.assertAlmostEqual(result["variance"], 0.02 / 3)
        self.assertAlmostEqual(result["std"] ** 2, result["variance"])
        self.assertEqual(result["n"], 3)

    def stores(self):
        stores = []
        for snr in (-6, 0, 6):
            config = {key: 1 for key in FRONTEND_KEYS}
            config["snr_db"] = snr
            row = {"relative_path": "id_00/normal/000.wav", "machine_id": "id_00", "label": 0,
                   "id_class": 0, "split": "train", "waveform_sha256": str(snr),
                   "snr_db": snr, "feature_path": "logmel/id_00/normal/000.npy"}
            stores.append(SimpleNamespace(config=config, records=[row], root=Path(f"features/{snr}")))
        return stores

    def test_versions_retain_group_identity_and_unique_paths(self):
        rows = combine_records(self.stores(), Path("features/combined"))
        self.assertEqual(len({r["relative_path"] for r in rows}), 3)
        self.assertEqual(len({r["source_recording_key"] for r in rows}), 1)
        rows[1]["split"] = "test"
        with self.assertRaisesRegex(ValueError, "Noise versions"):
            validate_splits(rows)

    def test_mismatched_splits_labels_or_frontends_are_rejected(self):
        for target, key, value in (("records", "split", "test"), ("records", "label", 1),
                                   ("config", "n_fft", 2048)):
            stores = deepcopy(self.stores())
            if target == "records":
                stores[1].records[0][key] = value
            else:
                stores[1].config[key] = value
            with self.assertRaises(ValueError):
                combine_records(stores, Path("features/combined"))

## Full-corpus feature stores

In [ ]:
__file__ = str(root / "tests/test_full_preprocessing.py")

In [ ]:
"""Full-corpus cache validation and cross-machine/SNR split isolation."""

import contextlib
import io
import json
from pathlib import Path
import tempfile
import unittest
from unittest.mock import patch

import numpy as np
import soundfile as sf

from mimii.config import PreprocessConfig
from mimii.prepare_full import SNR_NAMES, audit_cache, run
from mimii.training_data import FeatureStore

class FullPreprocessingTests(unittest.TestCase):
    def test_full_pipeline_resume_and_corrupt_cache_detection(self):
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            data, output = root / "data", root / "output"
            rng = np.random.default_rng(17)
            for machine in ("fan", "valve"):
                for snr, name in SNR_NAMES.items():
                    for index in range(6):
                        condition = "abnormal" if index == 5 else "normal"
                        path = data / f"{machine}_{name}" / machine / "id_00" / condition / f"{index:08}.wav"
                        path.parent.mkdir(parents=True, exist_ok=True)
                        sf.write(path, rng.normal(0, 0.1, (1600, 2)), 16000, subtype="PCM_16")
            config = {"data_dir": str(data), "output_dir": str(output),
                      "machine_types": ["fan", "valve"], "snr_levels_db": [0, -6, 6],
                      "frontend": {"duration_seconds": 0.1, "n_mels": 16, "context_frames": 3, "workers": 1}}
            with patch("mimii.visualization.create_figures"), contextlib.redirect_stdout(io.StringIO()):
                summary = run(config)
            self.assertEqual(summary["recordings"], 36)
            self.assertEqual(summary["unique_source_recordings"], 12)
            self.assertEqual(summary["split_counts"], {"train": 18, "validation": 6, "test": 12})
            self.assertEqual(summary["machine_ids"], {"fan/id_00": 0, "valve/id_00": 1})
            store = FeatureStore(output, "train_per_mel")
            self.assertEqual({r["id_class"] for r in store.records}, {0, 1})
            self.assertTrue(all(r["split"] == "test" for r in store.records if r["label"]))
            for row in store.records:
                self.assertTrue(Path(row["source_path"]).is_file())
                self.assertTrue(np.isfinite(store.load(row)).all())
            # Re-running verifies completed caches without re-extracting audio.
            with patch("mimii.prepare_full.extract_subset", side_effect=AssertionError("unexpected extraction")), \
                    contextlib.redirect_stdout(io.StringIO()):
                resumed = run(config)
            self.assertEqual(resumed["manifest_sha256"], summary["manifest_sha256"])
            with self.assertRaisesRegex(ValueError, "configuration changed"):
                run({**config, "frontend": {**config["frontend"], "seed": 99}})
            subset = output / "subsets" / "fan_0db"
            frontend = PreprocessConfig(**json.loads((subset / "config.json").read_text()))
            row = FeatureStore(subset).records[0]
            target = subset / row["feature_path"]
            np.save(target, np.full((16, frontend.expected_frames), np.nan, dtype=np.float32))
            with self.assertRaisesRegex(ValueError, "Invalid feature array"):
                audit_cache(subset, frontend)

if __name__ == "__main__":
    unittest.main()

## Training and anomaly scores

In [ ]:
__file__ = str(root / "tests/test_training.py")

In [ ]:
"""Model, scoring and evaluation invariants for normal-only experiments."""

import contextlib
import hashlib
import io
import json
from pathlib import Path
import tempfile
import unittest
from unittest.mock import patch

import numpy as np
import torch
from torch import nn

from mimii.evaluation import detection_metrics, evaluate_scores
from mimii.models import CNN_MODELS, DenseAutoencoder, MachineIdCNN, ResidualBlock, TemporalStatisticsPool
from mimii.train import TrainingConfig, batches, run, score_recordings, known_id_anomaly_scores

class TrainingTests(unittest.TestCase):
    def setUp(self):
        torch.set_num_threads(1)

    def test_official_ae_parameter_count_and_shapes(self):
        model = DenseAutoencoder()
        trainable = sum(p.numel() for p in model.parameters())
        running_statistics = sum(layer.running_mean.numel() + layer.running_var.numel()
                                 for layer in model.modules() if isinstance(layer, nn.BatchNorm1d))
        self.assertEqual(trainable, 267928)
        self.assertEqual(trainable + running_statistics, 269992)
        x = torch.randn(5, 640)
        self.assertEqual(model(x).shape, x.shape)
        self.assertEqual(MachineIdCNN()(torch.randn(5, 1, 128, 313)).shape, (5, 4))

    def test_metric_convention_and_direction(self):
        perfect = detection_metrics([0, 0, 1, 1], [0.1, 0.2, 0.8, 0.9], 0.5)
        self.assertEqual(perfect["auc"], 1)
        self.assertEqual(perfect["pauc"], 1)
        self.assertEqual((perfect["tn"], perfect["fp"], perfect["fn"], perfect["tp"]), (2, 0, 0, 2))
        chance = detection_metrics([0, 0, 1, 1], [1, 1, 1, 1], 0.5)
        self.assertEqual(chance["auc"], 0.5)
        self.assertEqual(chance["pauc"], 0.5)
        with self.assertRaises(ValueError):
            detection_metrics([0, 0], [0.1, 0.2], 0.5)

    def test_cnn_variants_support_both_frontends_and_backpropagation(self):
        for name, constructor in CNN_MODELS.items():
            for mel_bands in (64, 128):
                with self.subTest(model=name, mel_bands=mel_bands):
                    model = constructor(classes=4)
                    logits = model(torch.randn(2, 1, mel_bands, 313))
                    self.assertEqual(logits.shape, (2, 4))
                    nn.CrossEntropyLoss()(logits, torch.tensor([0, 1])).backward()
                    self.assertTrue(all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None))

    def test_residual_path_transmits_signal_with_zero_body(self):
        block = ResidualBlock(8, 8)
        for parameter in block.body.parameters():
            nn.init.zeros_(parameter)
        x = torch.rand(2, 8, 4, 4)
        torch.testing.assert_close(block(x), x)

    def test_temporal_statistics_distinguish_variation_and_have_finite_gradients(self):
        pool = TemporalStatisticsPool()
        steady = torch.ones(1, 1, 4, 4, requires_grad=True)
        varying = torch.tensor([0., 2., 0., 2.]).reshape(1, 1, 1, 4).expand(1, 1, 4, 4)
        a, b = pool(steady), pool(varying)
        torch.testing.assert_close(a[:, :1], b[:, :1])
        self.assertTrue((b[:, 1:] > a[:, 1:]).all())
        a.sum().backward()
        self.assertTrue(torch.isfinite(steady.grad).all())

    def test_threshold_uses_training_scores_only(self):
        test = [{"relative_path": f"{i}.wav", "machine_id": "id_00", "split": "test", "label": i}
                for i in (0, 1)]
        train = [{"relative_path": f"train{i}.wav", "machine_id": "id_00", "split": "train", "label": 0}
                 for i in (0, 1)]
        with tempfile.TemporaryDirectory() as directory, patch("mimii.evaluation.create_evaluation_figures"):
            a = evaluate_scores(test, [1., 20.], train, [0., 2.], directory)
            b = evaluate_scores(test, [3., 200.], train, [0., 2.], directory)
            self.assertAlmostEqual(a["per_id"]["id_00"]["threshold"], 1.9)
            self.assertEqual(a["per_id"]["id_00"]["threshold"], b["per_id"]["id_00"]["threshold"])
            train[0]["label"] = 1
            with self.assertRaises(ValueError):
                evaluate_scores(test, [1., 20.], train, [0., 2.], directory)

    def test_ae_averages_window_errors_to_one_clip_score(self):
        class Store:
            context_frames = 2
            def load(self, row):
                return np.full((2, 4), row["value"], dtype=np.float32)
        class ZeroReconstruction(nn.Module):
            def forward(self, x):
                return torch.zeros_like(x)
        scores = score_recordings(ZeroReconstruction(), Store(), [{"value": 2}, {"value": 3}],
                                  torch.device("cpu"), "autoencoder")
        np.testing.assert_allclose(scores, [4, 9])

    def test_cnn_scores_known_id_even_when_wrong_class_is_confident(self):
        class Store:
            def clip_array(self, rows):
                return np.zeros((len(rows), 1, 2, 2), dtype=np.float32)
        class WrongConfidentClassifier(nn.Module):
            def forward(self, x):
                return torch.tensor([[-5., 5.]]).repeat(len(x), 1)
        scores = score_recordings(WrongConfidentClassifier(), Store(), [{"id_class": 0}, {"id_class": 1}],
                                  torch.device("cpu"), "machine_id_cnn")
        self.assertGreater(scores[0], 9)
        self.assertLess(scores[1], 0.001)

    def test_known_id_score_preserves_tiny_probabilities_without_overflow(self):
        logits = torch.tensor([[50., 0., 0., 0.], [0., 1000., 0., 0.]])
        scores = known_id_anomaly_scores(logits, torch.tensor([0, 0]))
        self.assertGreater(scores[0], 0)
        self.assertAlmostEqual(float(scores[0] / (3 * np.exp(-50))), 1.0, places=12)
        self.assertTrue(torch.isfinite(scores).all())
        self.assertAlmostEqual(float(scores[1]), 1000)
        ordinary = torch.randn(6, 4, dtype=torch.float64)
        targets = torch.tensor([0, 1, 2, 3, 0, 1])
        expected = -ordinary.log_softmax(dim=1)[torch.arange(6), targets]
        torch.testing.assert_close(known_id_anomaly_scores(ordinary, targets), expected)

    def test_singleton_batch_is_merged_without_dropping_samples(self):
        indices = batches(9, 4)
        self.assertEqual([len(part) for part in indices], [4, 5])
        torch.testing.assert_close(torch.cat(indices), torch.arange(9))

    def test_training_end_to_end_on_small_feature_store(self):
        with tempfile.TemporaryDirectory() as directory:
            root = Path(directory)
            feature_root = root / "features"
            feature_root.mkdir()
            rng = np.random.default_rng(8)
            rows = []
            for mid, id_class in (("id_00", 0), ("id_02", 1)):
                for index, (split, label) in enumerate((("train", 0), ("train", 0), ("train", 0),
                                                        ("validation", 0), ("test", 0), ("test", 1))):
                    name = f"{mid}_{index}.npy"
                    np.save(feature_root / name, rng.normal(size=(16, 32)).astype(np.float32))
                    rows.append({"relative_path": name.replace(".npy", ".wav"), "feature_path": name,
                                 "machine_id": mid, "id_class": id_class, "label": label, "split": split,
                                 "waveform_sha256": name, "feature_shape": [16, 32]})
            manifest = "".join(json.dumps(r) + "\n" for r in rows)
            (feature_root / "manifest.jsonl").write_text(manifest)
            digest = hashlib.sha256(manifest.encode()).hexdigest()
            (feature_root / "status.json").write_text(json.dumps({"state": "complete", "manifest_sha256": digest}))
            (feature_root / "config.json").write_text(json.dumps({"n_mels": 16, "context_frames": 5, "seed": 42}))
            (feature_root / "summary.json").write_text(json.dumps({"machine_ids": {"id_00": 0, "id_02": 1}}))
            config = TrainingConfig(feature_dir=str(feature_root), output_dir=str(root / "run"),
                                    seeds=(1,), epochs=2, batch_size=32, cpu_threads=1, device="cpu")
            with patch("mimii.evaluation.create_evaluation_figures"), patch("mimii.train.plot_history"), contextlib.redirect_stdout(io.StringIO()):
                result = run(config)
            self.assertTrue(np.isfinite(result["macro_auc_mean"]))
            output = root / "run" / "seed_1"
            self.assertEqual(json.loads((output / "status.json").read_text())["state"], "complete")
            self.assertTrue((output / "model.pt").exists())
            self.assertEqual(len((output / "scores.csv").read_text().splitlines()), 5)
            with self.assertRaises(FileExistsError), contextlib.redirect_stdout(io.StringIO()):
                run(config)

if __name__ == "__main__":
    unittest.main()

## MobileNet weight initialization

In [ ]:
__file__ = str(root / "tests/test_mobilenet.py")

In [ ]:
"""Transfer-learning invariants: RGB adaptation, freezing, and exact CPU resume."""

import contextlib
from dataclasses import replace
import io
import json
from pathlib import Path
import tempfile
from types import SimpleNamespace
import unittest
from unittest.mock import patch

import numpy as np
import torch
from torch import nn
from torchvision.models import mobilenet_v3_small

from mimii.finetune import FinetuneConfig, epoch_pass, train_seed
from mimii.mobilenet import build_mobilenet, convert_first_convolution, make_optimizer, weights_sha256

class TinyTransfer(nn.Module):
    def __init__(self, classes=2):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1, bias=False), nn.BatchNorm2d(4), nn.ReLU())
        self.classifier = nn.Sequential(nn.Linear(4, classes))

    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=(2, 3)))

class MobileNetTests(unittest.TestCase):
    def setUp(self):
        torch.set_num_threads(1)

    def test_summed_kernels_match_replicated_input_and_native_shape(self):
        model = mobilenet_v3_small(weights=None)
        x = torch.randn(2, 1, 128, 313)
        expected = model.features[0][0](x.repeat(1, 3, 1, 1))
        convert_first_convolution(model)
        torch.testing.assert_close(model.features[0][0](x), expected, rtol=1e-5, atol=1e-5)
        adapted = build_mobilenet(4)
        self.assertEqual(adapted(x).shape, (2, 4))
        self.assertEqual(sum(p.numel() for p in adapted.parameters()), 1521668)

    def test_warmup_freezes_backbone_parameters_and_running_statistics(self):
        model = TinyTransfer()
        config = FinetuneConfig(device="cpu", batch_size=4, mixed_precision=False)
        x, y = torch.randn(8, 1, 8, 16), torch.arange(8) % 2
        before = {k: v.clone() for k, v in model.features.state_dict().items()}
        head_before = model.classifier[0].weight.detach().clone()
        optimizer = make_optimizer(model, config, True)
        epoch_pass(model, x, y, config, torch.device("cpu"), optimizer, True)
        for key, value in model.features.state_dict().items():
            torch.testing.assert_close(value, before[key], rtol=0, atol=0)
        self.assertFalse(torch.equal(head_before, model.classifier[0].weight))
        optimizer = make_optimizer(model, config, False)
        epoch_pass(model, x, y, config, torch.device("cpu"), optimizer, False)
        self.assertFalse(torch.equal(before["0.weight"], model.features[0].weight))
        self.assertFalse(torch.equal(before["1.running_mean"], model.features[1].running_mean))

    def test_weight_integrity_rejects_wrong_file(self):
        with tempfile.TemporaryDirectory() as temporary:
            path = Path(temporary) / "weights.pth"
            path.write_bytes(b"not official weights")
            with self.assertRaisesRegex(ValueError, "official PyTorch hash"):
                weights_sha256(path)

    def test_interrupted_training_resumes_across_unfreeze_boundary(self):
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            np.savez(root / "normalizer.npz", mean=np.zeros(8), std=np.ones(8))
            rng = np.random.default_rng(3)
            records, arrays = [], {}
            for id_class, mid in enumerate(("id_00", "id_02")):
                for snr in (-6, 0, 6):
                    for index, (split, label) in enumerate((("train", 0), ("train", 0), ("validation", 0), ("test", 0), ("test", 1))):
                        path = f"{snr}/{mid}/{index}.wav"
                        records.append({"relative_path": path, "machine_id": mid, "id_class": id_class,
                                        "split": split, "label": label, "snr_db": snr})
                        arrays[path] = rng.normal(size=(8, 16)).astype(np.float32)
            store = SimpleNamespace(root=root, manifest_sha256="fixture", config={},
                summary={"machine_ids": {"id_00": 0, "id_02": 1}}, mean=np.zeros(8, np.float32), std=np.ones(8, np.float32))
            store.split = lambda split: [r for r in records if r["split"] == split]
            store.load = lambda row: arrays[row["relative_path"]]
            store.clip_array = lambda rows: np.stack([store.load(r) for r in rows])[:, None]
            train, val = store.split("train"), store.split("validation")
            inputs = (torch.from_numpy(store.clip_array(train)), torch.tensor([r["id_class"] for r in train]),
                      torch.from_numpy(store.clip_array(val)), torch.tensor([r["id_class"] for r in val]))
            config = FinetuneConfig(output_dir=str(root / "complete"), machine_types=("fan",), seeds=(7,),
                epochs=2, warmup_epochs=1, patience=2, batch_size=4, evaluation_batch_size=4,
                device="cpu", mixed_precision=False)
            resumed_config = replace(config, output_dir=str(root / "resumed"))
            calls = 0
            def interrupt(*args, **kwargs):
                nonlocal calls
                calls += 1
                if calls == 3:
                    raise RuntimeError("simulated interruption")
                return epoch_pass(*args, **kwargs)
            with patch("mimii.finetune.build_mobilenet", side_effect=lambda classes, path: TinyTransfer(classes)), \
                    patch("mimii.finetune.weights_sha256", return_value="fixture"), \
                    patch("mimii.finetune.plot_history"), patch("mimii.evaluation.create_evaluation_figures"), \
                    contextlib.redirect_stdout(io.StringIO()):
                complete = train_seed(config, "fan", 7, store, torch.device("cpu"), inputs)
                with patch("mimii.finetune.epoch_pass", side_effect=interrupt), self.assertRaisesRegex(RuntimeError, "simulated"):
                    train_seed(resumed_config, "fan", 7, store, torch.device("cpu"), inputs)
                resumed = train_seed(resumed_config, "fan", 7, store, torch.device("cpu"), inputs)
                self.assertEqual(complete["cells"], resumed["cells"])
                self.assertEqual(complete["selected_epoch"], resumed["selected_epoch"])
                first = torch.load(root / "complete/fan/seed_7/last.pt", weights_only=False)
                second = torch.load(root / "resumed/fan/seed_7/last.pt", weights_only=False)
                for key in first["model_state"]:
                    torch.testing.assert_close(first["model_state"][key], second["model_state"][key], rtol=0, atol=0)
                repeated = train_seed(resumed_config, "fan", 7, store, torch.device("cpu"), inputs)
                self.assertEqual(repeated, resumed)

if __name__ == "__main__":
    unittest.main()

## Backbone selection and fine-tuning

In [ ]:
__file__ = str(root / "tests/test_cnn_optimization.py")

In [ ]:
"""CNN adaptation and validation-only optimization/resume invariants."""

import contextlib
import io
import json
from pathlib import Path
import tempfile
import time
from types import SimpleNamespace
import unittest
from unittest.mock import patch

import numpy as np
import torch
from torch import nn

from mimii.cnn_optimization import BudgetExpired, choose_trials, evaluate, fit, score_inputs
from mimii.report_cnn_optimization import audit
from mimii.mobilenet import set_training_stage
from mimii.transfer_models import build_transfer, grayscale_conv

class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.BatchNorm2d(4), nn.ReLU())
        self.classifier = nn.Sequential(nn.Dropout(.2), nn.Linear(4, 2))

    def forward(self, x):
        return self.classifier(self.features(x).mean((2, 3)))

class CNNOptimizationTests(unittest.TestCase):
    def setUp(self):
        torch.set_num_threads(1)

    def test_grayscale_adaptation_matches_rgb_replication(self):
        original = nn.Conv2d(3, 8, 3, stride=2, padding=1, bias=True)
        x = torch.randn(2, 1, 19, 23)
        torch.testing.assert_close(grayscale_conv(original)(x), original(x.repeat(1, 3, 1, 1)))

    def test_backbones_accept_native_spectrograms_and_freeze_batchnorm(self):
        for architecture, parameters in (("resnet18", 11172292), ("efficientnet_b0", 4012096)):
            model = build_transfer(architecture, pretrained=False)
            self.assertEqual(sum(p.numel() for p in model.parameters()), parameters)
            set_training_stage(model, True)
            self.assertFalse(model.features.training)
            self.assertTrue(all(not p.requires_grad for p in model.features.parameters()))
            self.assertEqual(model(torch.randn(2, 1, 128, 313)).shape, (2, 4))
            set_training_stage(model, False)
            self.assertTrue(all(p.requires_grad for p in model.features.parameters()))

    def test_selection_uses_macro_validation_not_fault_metrics(self):
        rows = [{"architecture": "resnet18", "trial": trial, "machine_type": machine,
                 "validation_loss": loss, "macro_auc": 1 - loss}
                for trial, losses in ((0, (.1, .8)), (1, (.3, .3))) for machine, loss in zip(("fan", "pump"), losses)]
        selected = choose_trials(rows, ["resnet18"], ["fan", "pump"], 2)
        self.assertEqual(selected["resnet18"]["selected_trial"], 1)
        for row in rows:
            row["macro_auc"] = -100 * row["macro_auc"]
        self.assertEqual(choose_trials(rows, ["resnet18"], ["fan", "pump"], 2), selected)
        with self.assertRaises(ValueError):
            choose_trials(rows[:-1], ["resnet18"], ["fan", "pump"], 2)

    def test_staged_fit_resume_exact_and_no_test_access(self):
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            study = {"feature_root": "unused", "output_dir": str(root / "full"), "seed": 42,
                     "backbone_learning_rates": [1e-4], "head_lr_multiplier": 3, "final_epochs": 3,
                     "warmup_epochs": 1, "batch_sizes": {"resnet18": 4}, "weight_decay": 1e-4,
                     "threshold_quantile": .95, "device": "cpu"}
            store = SimpleNamespace(summary={"machine_ids": {"id_00": 0, "id_02": 1}},
                                    mean=np.zeros(8, np.float32), std=np.ones(8, np.float32), config={}, manifest_sha256="fixture")
            torch.manual_seed(13)
            inputs = (torch.randn(12, 1, 8, 16), torch.arange(12) % 2, torch.randn(6, 1, 8, 16), torch.arange(6) % 2)
            with patch("mimii.cnn_optimization.build_transfer", side_effect=lambda *a, **k: TinyCNN()), contextlib.redirect_stdout(io.StringIO()):
                full = fit(study, "resnet18", 0, "fan", store, inputs, 3, time.time() + 100)
                resumed_study = {**study, "output_dir": str(root / "resumed")}
                fit(resumed_study, "resnet18", 0, "fan", store, inputs, 1, time.time() + 100)
                resumed = fit(resumed_study, "resnet18", 0, "fan", store, inputs, 3, time.time() + 100)
                with self.assertRaises(BudgetExpired):
                    fit({**study, "output_dir": str(root / "expired")}, "resnet18", 0, "fan", store, inputs, 3, time.time() - 1)
            self.assertEqual(full["validation_loss"], resumed["validation_loss"])
            self.assertEqual(full["selected_epoch"], resumed["selected_epoch"])
            left = torch.load(root / "full/resnet18/lr_0/fan/last.pt", weights_only=False)
            right = torch.load(root / "resumed/resnet18/lr_0/fan/last.pt", weights_only=False)
            for key in left["model_state"]:
                torch.testing.assert_close(left["model_state"][key], right["model_state"][key], rtol=0, atol=0)
            self.assertFalse((root / "full/resnet18/lr_0/fan/scores.csv").exists())

    def test_known_id_scoring_is_batch_size_independent(self):
        model = TinyCNN().eval()
        x, y = torch.randn(7, 1, 8, 16), torch.arange(7) % 2
        np.testing.assert_allclose(score_inputs(model, x, y, 2), score_inputs(model, x, y, 7), rtol=1e-6)

    def test_selected_fit_scoring_and_saved_result_audit(self):
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            study = {"feature_root": "unused", "output_dir": str(root), "seed": 42,
                     "architectures": ["resnet18"], "machine_types": ["fan"],
                     "backbone_learning_rates": [1e-4], "head_lr_multiplier": 3, "final_epochs": 2,
                     "screening_epochs": 2, "warmup_epochs": 1, "batch_sizes": {"resnet18": 4},
                     "weight_decay": 1e-4, "threshold_quantile": .95, "device": "cpu"}
            rng = np.random.default_rng(8)
            records, arrays = [], {}
            for snr in (-6, 0, 6):
                for cls, mid in enumerate(("id_00", "id_02")):
                    for index, (split, label) in enumerate((("train", 0), ("validation", 0), ("test", 0), ("test", 1))):
                        path = f"{snr}/{mid}/{index}.wav"
                        records.append({"relative_path": path, "machine_id": mid, "id_class": cls,
                                        "split": split, "label": label, "snr_db": snr})
                        arrays[path] = rng.normal(size=(8, 16)).astype(np.float32)
            store = SimpleNamespace(summary={"machine_ids": {"id_00": 0, "id_02": 1}},
                                    mean=np.zeros(8, np.float32), std=np.ones(8, np.float32), config={}, manifest_sha256="fixture")
            store.split = lambda split: [r for r in records if r["split"] == split]
            store.clip_array = lambda rows: np.stack([arrays[r["relative_path"]] for r in rows])[:, None]
            inputs = tuple(value for split in ("train", "validation") for value in
                           (torch.from_numpy(store.clip_array(store.split(split))), torch.tensor([r["id_class"] for r in store.split(split)])))
            with patch("mimii.cnn_optimization.build_transfer", side_effect=lambda *a, **k: TinyCNN()), \
                    patch("mimii.evaluation.create_evaluation_figures"), patch("mimii.cnn_optimization.plot_history"), \
                    contextlib.redirect_stdout(io.StringIO()):
                fitted = fit(study, "resnet18", 0, "fan", store, inputs, 2, time.time() + 100)
                result = evaluate(study, "resnet18", 0, "fan", store, inputs)
            selection = choose_trials([fitted], ["resnet18"], ["fan"], 1)
            for name, value in (("protocol.json", {"identity": {"source_sha256": {}}}), ("selection.json", selection), ("screening.json", [fitted])):
                (root / name).write_text(json.dumps(value))
            checked = audit(root, study, [result], {"fan": store})
            self.assertEqual(checked["state"], "passed")
            self.assertEqual(checked["checked_test_scores"], 12)
            scores = root / "resnet18/lr_0/fan/scores.csv"
            scores.write_text(scores.read_text().replace("id_00", "id_99", 1))
            with self.assertRaisesRegex(ValueError, "Saved scores do not cover exact split|Machine ID mismatch"):
                audit(root, study, [result], {"fan": store})

## Checkpoint continuation

In [ ]:
__file__ = str(root / "tests/test_cnn_continuation.py")

In [ ]:
"""Continuation preserves parent files and rejects changed provenance."""

import hashlib
import importlib.util
from importlib.metadata import version
import json
from pathlib import Path
import tempfile
import unittest
from unittest.mock import patch


SCRIPT = Path(__file__).resolve().parents[1] / "scripts/extend_cnn_study.py"
spec = importlib.util.spec_from_file_location("extend_cnn_study", SCRIPT)
continuation = importlib.util.module_from_spec(spec)
spec.loader.exec_module(continuation)

class ContinuationTests(unittest.TestCase):
    def test_copy_preserves_parent_and_resumes_with_original_deadline(self):
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            parent, output, features = root / "parent", root / "extended", root / "features"
            parent.mkdir()
            normalizer = features / "by_machine/fan/normalizer.npz"
            normalizer.parent.mkdir(parents=True)
            normalizer.write_bytes(b"normalization fixture")
            (features / "manifest.jsonl").write_bytes(b"manifest fixture")
            study = {"feature_root": str(features), "output_dir": str(parent), "architectures": ["resnet18"],
                     "machine_types": ["fan"], "final_epochs": 6, "seed": 42}
            identity = {"study": study, "source_sha256": {},
                        "versions": {name: version(name) for name in ("torch", "torchvision", "numpy", "scikit-learn")},
                        "pretrained_sha256": {"resnet18": "pretrained fixture"},
                        "master_manifest_sha256": continuation.digest(features / "manifest.jsonl"),
                        "normalizer_sha256": {"fan": continuation.digest(normalizer)}}
            for name, value in (("protocol.json", {"identity": identity}), ("status.json", {"state": "complete"}),
                                ("validation.json", {"state": "passed"}), ("final_plan.json", {"target_epochs": 6}),
                                ("selection.json", {"resnet18": {"selected_trial": 1}}), ("screening.json", []), ("summary.json", {})):
                (parent / name).write_text(json.dumps(value))
            for index in (0, 1):
                directory = parent / f"resnet18/lr_{index}/fan"
                directory.mkdir(parents=True)
                (directory / "history.json").write_text("[]\n")
            directory = parent / "resnet18/lr_1/fan"
            for name in ("last.pt", "model.pt", "fit.json"):
                (directory / name).write_bytes(b"parent checkpoint fixture")
            with patch.object(continuation, "pretrained_hash", return_value="pretrained fixture"):
                first = continuation.prepare(parent, output, 12, 75)
                second = continuation.prepare(parent, output, 12, 75)
                self.assertEqual(first[2]["deadline_unix"], second[2]["deadline_unix"])
                self.assertEqual(first[0]["final_epochs"], 12)
                self.assertEqual(first[2]["deadline_unix"] - first[2]["started_unix"], 4500)
                copied = output / "resnet18/lr_1/fan/last.pt"
                self.assertNotEqual(copied.stat().st_ino, (directory / "last.pt").stat().st_ino)
                copied.write_bytes(b"continued checkpoint")
                self.assertEqual((directory / "last.pt").read_bytes(), b"parent checkpoint fixture")
                continuation.prepare(parent, output, 12, 75)
                self.assertEqual(copied.read_bytes(), b"continued checkpoint")
                with self.assertRaisesRegex(ValueError, "settings changed"):
                    continuation.prepare(parent, output, 13, 75)
                with self.assertRaisesRegex(ValueError, "extend the parent"):
                    continuation.prepare(parent, root / "invalid", 6, 75)

## Compact CNN and pooling

In [ ]:
__file__ = str(root / "tests/test_acoustic_cnn.py")

In [ ]:
"""Custom-model shape, pooling hypothesis, scratch-training and resume checks."""

import contextlib
from dataclasses import replace
import io
from pathlib import Path
import tempfile
from types import SimpleNamespace
import unittest
from unittest.mock import patch

import numpy as np
import torch

from mimii.acoustic_cnn import AcousticResidualStatsCNN, FrequencyStatisticsPool
from mimii.custom_cnn_experiment import AcousticConfig
from mimii.failure_analysis import choose_microphone_sample
from mimii.finetune import train_seed

class AcousticCNNTests(unittest.TestCase):
    def setUp(self):
        torch.set_num_threads(1)

    def test_native_shape_and_finite_gradients_in_both_variants(self):
        for enabled in (True, False):
            model = AcousticResidualStatsCNN(include_std=enabled)
            self.assertEqual(sum(p.numel() for p in model.parameters()), 177724)
            x = torch.randn(2, 1, 128, 313)
            self.assertEqual(model.features(x).shape, (2, 80, 4, 20))
            logits = model(x)
            self.assertEqual(logits.shape, (2, 4))
            torch.nn.functional.cross_entropy(logits, torch.tensor([0, 1])).backward()
            self.assertTrue(all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None))

    def test_std_pool_distinguishes_variation_with_equal_mean(self):
        steady = torch.ones(2, 3, 4, 8, requires_grad=True)
        varying = torch.tensor([0., 2.] * 4).expand(2, 3, 4, 8)
        full, mean_only = FrequencyStatisticsPool(True), FrequencyStatisticsPool(False)
        a, b = full(steady), full(varying)
        torch.testing.assert_close(a[:, :3], b[:, :3])
        self.assertTrue((b[:, 3:] > a[:, 3:]).all())
        torch.testing.assert_close(mean_only(steady), mean_only(varying))
        self.assertEqual(torch.count_nonzero(mean_only(steady)[:, 3:]).item(), 0)
        a.sum().backward()
        self.assertTrue(torch.isfinite(steady.grad).all())

    def test_sample_selection_is_score_independent_and_source_paired(self):
        rows = [{"machine_id": mid, "label": label, "snr_db": snr, "split": "test",
                 "source_recording_key": f"{mid}/{label}/{i}", "relative_path": f"{snr}/{mid}/{label}/{i}"}
                for mid in ("id_00", "id_02") for label in (0, 1) for i in range(6) for snr in (-6, 0, 6)]
        a = choose_microphone_sample(rows, 3)
        b = choose_microphone_sample(list(reversed(rows)), 3)
        self.assertEqual(a, b)
        self.assertEqual(len(a), 36)
        with self.assertRaises(ValueError):
            choose_microphone_sample(rows, 7)

    def test_random_factory_has_no_pretrained_dependency_and_resumes(self):
        # Exercise the production trainer's new factory path on real custom layers.
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            np.savez(root / "normalizer.npz", mean=np.zeros(128), std=np.ones(128))
            rng = np.random.default_rng(9)
            records, arrays = [], {}
            for snr in (-6, 0, 6):
                for cls, mid in enumerate(("id_00", "id_02")):
                    for i, (split, label) in enumerate((("train", 0), ("validation", 0), ("test", 0), ("test", 1))):
                        path = f"{snr}/{mid}/{i}.wav"
                        records.append({"relative_path": path, "machine_id": mid, "id_class": cls, "snr_db": snr, "split": split, "label": label})
                        arrays[path] = rng.normal(size=(128, 64)).astype(np.float32)
            store = SimpleNamespace(root=root, manifest_sha256="fixture", config={}, summary={"machine_ids": {"id_00": 0, "id_02": 1}},
                                    mean=np.zeros(128, np.float32), std=np.ones(128, np.float32))
            store.split = lambda split: [r for r in records if r["split"] == split]
            store.clip_array = lambda rows: np.stack([arrays[r["relative_path"]] for r in rows])[:, None]
            inputs = tuple(v for split in ("train", "validation") for v in
                           (torch.from_numpy(store.clip_array(store.split(split))), torch.tensor([r["id_class"] for r in store.split(split)])))
            cfg = AcousticConfig(output_dir=str(root / "run"), pretrained_path="absent", seeds=(42,), epochs=2,
                                 warmup_epochs=0, patience=5, batch_size=3, evaluation_batch_size=3, device="cpu", mixed_precision=False)
            with patch("mimii.finetune.plot_history"), patch("mimii.evaluation.create_evaluation_figures"), contextlib.redirect_stdout(io.StringIO()):
                result = train_seed(cfg, "fan", 42, store, torch.device("cpu"), inputs,
                                    model_factory=lambda n: AcousticResidualStatsCNN(n), model_name="custom_test")
                repeated = train_seed(cfg, "fan", 42, store, torch.device("cpu"), inputs,
                                      model_factory=lambda n: AcousticResidualStatsCNN(n), model_name="custom_test")
            self.assertEqual(result, repeated)
            self.assertEqual(result["model"], "custom_test")
            checkpoint = torch.load(root / "run/fan/seed_42/model.pt", weights_only=False)
            self.assertIsNone(checkpoint["config"]["pretrained_sha256"])
            self.assertEqual(checkpoint["config"]["initialization"], "random, no pretrained weights")

## Wide CNN and detection thresholds

In [ ]:
__file__ = str(root / "tests/test_large_acoustic_cnn.py")

In [ ]:
"""Validate wider-model training coverage and comparison metric semantics."""

from dataclasses import asdict
import json
from pathlib import Path
import unittest

import torch

from mimii.custom_cnn_experiment import variant_config as small_config
from mimii.large_acoustic_cnn import LargeAcousticResidualStatsCNN
from mimii.large_cnn_experiment import operating_point, variant_config
from mimii.mobilenet import make_optimizer

class LargeAcousticTests(unittest.TestCase):
    def setUp(self):
        torch.set_num_threads(1)

    def test_forward_backward_optimizer_coverage_and_checkpoint(self):
        model = LargeAcousticResidualStatsCNN()
        self.assertEqual(sum(p.numel() for p in model.parameters()), 1286964)
        study = json.loads(Path("configs/train_large_acoustic_cnn.json").read_text())
        optimizer = make_optimizer(model, variant_config(study), False)
        optimized = [p for group in optimizer.param_groups for p in group["params"]]
        self.assertEqual(len(optimized), len(set(map(id, optimized))))
        self.assertEqual(set(map(id, optimized)), set(map(id, model.parameters())))
        x = torch.randn(2, 1, 128, 313)
        self.assertEqual(model.features(x).shape, (2, 256, 4, 20))
        logits = model(x)
        self.assertEqual(logits.shape, (2, 4))
        torch.nn.functional.cross_entropy(logits, torch.tensor([0, 1])).backward()
        self.assertTrue(all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters()))
        optimizer.step()
        restored = LargeAcousticResidualStatsCNN()
        restored.load_state_dict(model.state_dict())
        model.eval()
        restored.eval()
        with torch.no_grad():
            torch.testing.assert_close(model(x), restored(x), rtol=0, atol=0)

    def test_training_settings_match_small_model(self):
        small = asdict(small_config(json.loads(Path("configs/train_acoustic_cnn.json").read_text()), True))
        large = asdict(variant_config(json.loads(Path("configs/train_large_acoustic_cnn.json").read_text())))
        large.pop("architecture")
        small.pop("output_dir")
        large.pop("output_dir")
        self.assertEqual(small, large)

    def test_operating_point_counts_clips_instead_of_averaging_cells(self):
        actual = operating_point([{"tp": 1, "fn": 0, "fp": 0, "tn": 1},
                                  {"tp": 0, "fn": 9, "fp": 9, "tn": 0}])
        self.assertEqual(actual["fault_recall"], .1)
        self.assertEqual(actual["false_positive_rate"], .9)

## Autoencoder context windows

In [ ]:
__file__ = str(root / "tests/test_ae_windows.py")

In [ ]:
"""Window isolation, full coverage, clip scoring and exact CPU resume."""

import copy
import contextlib
from dataclasses import replace
import io
import json
import os
from pathlib import Path
import tempfile
from types import SimpleNamespace
import unittest
from unittest.mock import patch

import numpy as np
import torch

from mimii.ae_experiment import AEConfig, assert_normal_splits, train_seed, summarize
from mimii.ae_report import audit_run, publish_report
from mimii.ae_windows import WindowBank, batch_slices, score_bank, train_epoch
from mimii.features import context_vectors
from mimii.models import DenseAutoencoder

class AEWindowTests(unittest.TestCase):
    def setUp(self):
        torch.set_num_threads(1)

    def test_all_windows_match_reference_without_crossing_clips(self):
        features = torch.arange(3 * 4 * 9, dtype=torch.float32).reshape(3, 4, 9)
        bank = WindowBank(features, 5)
        expected = np.concatenate([context_vectors(x.numpy(), 5) for x in features])
        order = torch.tensor([14, 0, 5, 4, 9, 10, 3, 7, 2, 11, 1, 6, 8, 12, 13])
        np.testing.assert_array_equal(bank.batch(order).numpy(), expected[order.numpy()])

    def test_full_coverage_including_singleton_tail(self):
        for count in (2, 3, 511, 512, 513, 1024, 1025, 1537):
            slices = list(batch_slices(count, 512))
            self.assertTrue(all(stop - start >= 2 for start, stop in slices))
            self.assertEqual([i for start, stop in slices for i in range(start, stop)], list(range(count)))

    def test_scores_average_features_then_all_windows(self):
        features = torch.arange(2 * 3 * 8, dtype=torch.float32).reshape(2, 3, 8)
        bank = WindowBank(features, 5)
        zero = torch.nn.Linear(15, 15, bias=False)
        with torch.no_grad():
            zero.weight.zero_()
        scores = score_bank(zero, bank, batch_size=3)
        expected = np.array([(context_vectors(x.numpy(), 5) ** 2).mean() for x in features])
        np.testing.assert_allclose(scores, expected, rtol=1e-6)

    def test_epoch_resume_restores_shuffle_optimizer_and_batchnorm(self):
        torch.manual_seed(7)
        bank = WindowBank(torch.randn(3, 2, 9), 5)
        model = DenseAutoencoder(10, 3)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001, eps=1e-7)
        generator = torch.Generator().manual_seed(42)
        train_epoch(model, optimizer, bank, 4, generator)
        model_state, optimizer_state, rng = copy.deepcopy(model.state_dict()), copy.deepcopy(optimizer.state_dict()), generator.get_state()
        expected_loss = train_epoch(model, optimizer, bank, 4, generator)
        restored = DenseAutoencoder(10, 3)
        restored.load_state_dict(model_state)
        restored_optimizer = torch.optim.Adam(restored.parameters(), lr=0.001, eps=1e-7)
        restored_optimizer.load_state_dict(optimizer_state)
        restored_generator = torch.Generator().set_state(rng)
        loss = train_epoch(restored, restored_optimizer, bank, 4, restored_generator)
        self.assertEqual(loss, expected_loss)
        for key, tensor in model.state_dict().items():
            torch.testing.assert_close(tensor, restored.state_dict()[key], rtol=0, atol=0)

    def test_only_healthy_training_and_validation_are_accepted(self):
        train = [{"label": 0, "split": "train"}]
        validation = [{"label": 0, "split": "validation"}]
        assert_normal_splits(train, validation)
        for invalid in ([{"label": 1, "split": "train"}], [{"label": 0, "split": "test"}], []):
            with self.assertRaises(ValueError):
                assert_normal_splits(invalid, validation)

    def test_frozen_baseline_settings(self):
        config = AEConfig()
        self.assertEqual((config.epochs, config.batch_size, config.latent_dimensions), (100, 512, 8))
        for kwargs in ({"normalization": "train_per_mel"}, {"device": "cpu"}, {"batch_size": 1}, {"seeds": [42, 42]}):
            with self.assertRaises(ValueError):
                AEConfig(**kwargs)

    def test_runner_resume_and_both_evaluation_views(self):
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            rng = np.random.default_rng(5)
            records, arrays = [], {}
            for snr in (-6, 0, 6):
                for i, (split, label) in enumerate((("train", 0), ("validation", 0), ("test", 0), ("test", 1))):
                    path = f"{snr}/{i}.wav"
                    records.append({"relative_path": path, "machine_id": "id_00", "split": split, "label": label, "snr_db": snr})
                    arrays[path] = rng.normal(size=(2, 9)).astype(np.float32)
            store = SimpleNamespace(shape=(2, 9), context_frames=5, config={}, manifest_sha256="fixture")
            store.load = lambda row: arrays[row["relative_path"]]
            store.split = lambda split: [r for r in records if r["split"] == split]
            train = WindowBank.from_store(store, store.split("train"), "cpu")
            validation = WindowBank.from_store(store, store.split("validation"), "cpu")
            config = AEConfig(output_dir=str(root / "complete"), machine_types=("fan",), seeds=(42,), epochs=2,
                              device="cpu", cuda_graphs=False, batch_size=4, evaluation_batch_size=3)
            other = replace(config, output_dir=str(root / "resumed"))
            calls = 0
            def interrupt(*args, **kwargs):
                nonlocal calls
                calls += 1
                if calls == 2:
                    raise RuntimeError("simulated interruption")
                return train_epoch(*args, **kwargs)
            with patch("mimii.evaluation.create_evaluation_figures"), contextlib.redirect_stdout(io.StringIO()):
                full = train_seed(config, "fan", 42, store, train, validation, Path(config.output_dir))
                with patch("mimii.ae_experiment.train_epoch", side_effect=interrupt), self.assertRaisesRegex(RuntimeError, "simulated"):
                    train_seed(other, "fan", 42, store, train, validation, Path(other.output_dir))
                resumed = train_seed(other, "fan", 42, store, train, validation, Path(other.output_dir))
            for view in ("final", "best_validation"):
                self.assertEqual(full[view], resumed[view])
                checkpoint = torch.load(root / f"resumed/fan/seed_42/{view}.pt", weights_only=False)
                self.assertEqual(checkpoint["normalization"], "none")
            report = summarize(Path(other.output_dir), other)
            self.assertTrue(report["complete"])
            self.assertEqual(report["views"]["final"]["overall"]["auc"]["mean"], full["final"]["macro_auc"])
            self.assertEqual(train_seed(other, "fan", 42, store, train, validation, Path(other.output_dir)), resumed)
            incomplete = summarize(Path(other.output_dir), replace(other, seeds=(42, 43)))
            self.assertFalse(incomplete["complete"])
            self.assertEqual(incomplete["views"]["final"]["overall"], {})
            report = summarize(Path(other.output_dir), other)
            previous_directory = Path.cwd()
            try:
                os.chdir(root)
                (Path(other.output_dir) / "protocol.json").write_text("{}\n")
                publish_report(Path(other.output_dir), report, other)
                self.assertTrue(Path("docs/results/dcase2020_ae/artifact_hashes.json").exists())
                self.assertTrue(Path("docs/results/dcase2020_ae/ae_mobilenet_comparison.pdf").exists())
            finally:
                os.chdir(previous_directory)
            score_path = Path(other.output_dir) / "fan/seed_42/final/scores.csv"
            score_path.write_text(score_path.read_text().replace("id_00", "id_99", 1))
            with self.assertRaisesRegex(ValueError, "machine ID mismatch"):
                audit_run(Path(other.output_dir) / "fan/seed_42", store, other)

## Run the tests

In [ ]:
import subprocess

subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests"],
    env={**os.environ, "PYTHONPATH": str(root / "src")},
    check=True,
);